In [1]:
# 🟧 **smoke_test.ipynb (Notebook‑Version)**

## **Notebook‑Zelle 1 — Imports**

import pandas as pd
import pynonym

from pynonym import PynonymConfig
from pynonym.text import anonymize_text
from pynonym.tables import TableAnonymizationConfig, anonymize_dataframe
import spacy

print("Imports erfolgreich.")

Imports erfolgreich.


In [2]:
## **Notebook‑Zelle 2 — spaCy‑Modell**

nlp = spacy.load("de_core_news_md")
print("spaCy‑Modell geladen:", nlp.meta["name"])

spaCy‑Modell geladen: core_news_md


In [3]:
## **Notebook‑Zelle 3 — Text‑Anonymisierung**

cfg = PynonymConfig(language="de", seed=42)
text = "Angela Merkel traf Olaf Scholz in Berlin."
anon = anonymize_text(text, config=cfg)

print("Original:", text)
print("Anonymisiert:", anon)


Original: Angela Merkel traf Olaf Scholz in Berlin.
Anonymisiert: Angela Merkel traf Olaf Scholz in Aleksandr Weihmann.


In [4]:
## **Notebook‑Zelle 4 — Tabellen‑Anonymisierung**

df = pd.DataFrame({
    "Name": ["Angela Merkel", "Olaf Scholz", "Karl Lauterbach"],
    "Stadt": ["Berlin", "Hamburg", "Köln"],
    "Diagnose": ["A", "B", "A"]
})

tcfg = TableAnonymizationConfig(
    pseudonymize_columns=["Name"],
    quasi_identifiers=["Stadt"],
    sensitive_attributes=["Diagnose"],
    seed=42,
    k=2,
    l=1,
    t=0.5
)

df_anon = anonymize_dataframe(df, config=tcfg)
df_anon

Warnung: pycanon nicht verfügbar. k-Anonymität deaktiviert.
Warnung: pycanon nicht verfügbar. l-Diversität deaktiviert.
Warnung: pycanon nicht verfügbar. t-Closeness deaktiviert.


,Name,Stadt,Diagnose
0,Eleni Hauffer,Berlin,A
1,Paul Dobes-Stey,Hamburg,B
2,Klemens Löchel,Köln,A


In [5]:
## **Notebook‑Zelle 5 — Privacy‑Metriken**

df_anon.attrs

{'k_anonymity': {'metric': 'k-anonymity',
  'value': None,
  'status': 'pycanon_not_available',
  'message': 'k-anonymity ist unter Windows deaktiviert (pycanon nicht installiert).'},
 'l_diversity': {'metric': 'l-diversity',
  'value': None,
  'status': 'pycanon_not_available',
  'message': 'l-diversity ist unter Windows deaktiviert (pycanon nicht installiert).'},
 't_closeness': {'metric': 't-closeness',
  'value': None,
  'status': 'pycanon_not_available',
  'message': 't-closeness ist unter Windows deaktiviert (pycanon nicht installiert).'}}

In [6]:
## **Notebook‑Zelle 6 — Determinismus**

cfg2 = PynonymConfig(language="de", seed=42)
anon2 = anonymize_text(text, config=cfg2)
anon == anon2

True

In [10]:
# Notebook‑Zelle 7 - Eigene Privacy Metriken (k-Anonymität, l-Diversität, t-Closeness)
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
def k_anonymity(df: pd.DataFrame, qi: list, k: int):
    """
    Berechnet k-Anonymität:
    - Jede QI-Kombination muss mindestens k-mal vorkommen.
    """
    groups = df.groupby(qi).size()
    min_group_size = groups.min()
    return {
        "metric": "k-anonymity",
        "value": int(min_group_size),
        "status": "ok" if min_group_size >= k else "violated",
        "message": f"Minimale Gruppengröße = {min_group_size}, benötigt >= {k}"
    }

def l_diversity(df: pd.DataFrame, qi: list, sa: list, l: int):
    """
    Berechnet l-Diversität:
    - Jede QI-Gruppe muss mindestens l verschiedene sensitive Werte enthalten.
    """
    results = {}
    for attr in sa:
        diversities = df.groupby(qi)[attr].nunique()
        min_div = diversities.min()
        results[attr] = {
            "metric": "l-diversity",
            "attribute": attr,
            "value": int(min_div),
            "status": "ok" if min_div >= l else "violated",
            "message": f"Minimale Diversität für {attr} = {min_div}, benötigt >= {l}"
        }
    return results

def t_closeness(df: pd.DataFrame, qi: list, sa: list, t: float):
    """
    Berechnet t-Closeness:
    - Verteilung in jeder QI-Gruppe darf nicht mehr als t von der Gesamtverteilung abweichen.
    - Distanzmaß: Earth-Mover-Distance (vereinfachte Version).
    """
    results = {}
    for attr in sa:
        # Gesamtverteilung
        global_dist = df[attr].value_counts(normalize=True)
        group_results = {}
        for group_vals, group_df in df.groupby(qi):
            group_dist = group_df[attr].value_counts(normalize=True)
            # Earth-Mover-Distance (vereinfachte L1-Distanz)
            all_vals = set(global_dist.index) | set(group_dist.index)
            dist = sum(abs(global_dist.get(v, 0) - group_dist.get(v, 0)) for v in all_vals) / 2
            group_results[group_vals] = dist
        max_dist = max(group_results.values())
        results[attr] = {
            "metric": "t-closeness",
            "attribute": attr,
            "value": float(max_dist),
            "status": "ok" if max_dist <= t else "violated",
            "message": f"Maximale Distanz = {max_dist:.3f}, erlaubt <= {t}"
        }
    return results

# Beispiel: Metriken auf anonymisiertem DF berechnen
def compute_privacy_metrics(df, qi, sa, k=2, l=1, t=0.5):
    metrics = {}
    metrics["k_anonymity"] = k_anonymity(df, qi, k)
    metrics["l_diversity"] = l_diversity(df, qi, sa, l)
    metrics["t_closeness"] = t_closeness(df, qi, sa, t)
    return metrics
print("Privacy-Metriken-Modul geladen.")
 
############
 
# Privacy-Metriken auf df_anon anwenden
qi = ["Stadt"]
sa = ["Diagnose"]
metrics = compute_privacy_metrics(df_anon, qi=qi, sa=sa, k=2, l=1, t=0.5)
metrics


Privacy-Metriken-Modul geladen.


{'k_anonymity': {'metric': 'k-anonymity',
  'value': 1,
  'status': 'violated',
  'message': 'Minimale Gruppengröße = 1, benötigt >= 2'},
 'l_diversity': {'Diagnose': {'metric': 'l-diversity',
   'attribute': 'Diagnose',
   'value': 1,
   'status': 'ok',
   'message': 'Minimale Diversität für Diagnose = 1, benötigt >= 1'}},
 't_closeness': {'Diagnose': {'metric': 't-closeness',
   'attribute': 'Diagnose',
   'value': 0.6666666666666667,
   'status': 'violated',
   'message': 'Maximale Distanz = 0.667, erlaubt <= 0.5'}}}